# 募資專案關鍵字萃取（圖文多模態版）
**使用模型：** gemini-3.1-flash-lite（維持原本使用的模型不變）、gemini-3.5-flash-lite

本版在原本「純文字」流程的基礎上，新增以下功能：

1. **圖文多模態分析**：每個專案除了讀取 `*_content.txt` 之外，會依文字量決定是否一併讀取
   `*_images` 資料夾內的圖片（見下方第 4 點），將文字與圖片一起丟給 Gemini 分析。
2. **全資料夾掃描**：不再假設固定的 4 層資料夾結構，改用遞迴掃描
   `ZecZec Group Data` 底下所有子資料夾，只要該資料夾內含有 `*content.txt`，
   就視為一個「專案」。
3. **三層防禦機制**，確保長時間大量跑資料時不會卡死：
   - **第一層｜多金鑰輪替（Failover Key Pool）**：3 組 Gemini API Key 輪流使用，
     只要遇到 429 限速或連線逾時（Colab 網路環境常見），約 1.5 秒內自動無縫換下一把 Key 繼續。
   - **圖片降級**：同一把 Key 若因圖片 payload 太大導致逾時，會自動把圖片張數砍半再試
     （8→4→2→1→0 張），每個張數組合都會把所有 Key 各輪一次。
   - **第二層｜本地端傳統 NLP 備援（On-premise Classical NLP）**：所有 Key、所有圖片組合
     都打不通時，自動切換成本地 Jieba 斷詞 + TextRank 演算法，在完全離線狀態下抓取
     募資內文的高頻核心名詞作為關鍵字，維持關鍵字產出不中斷，並在結果中標記
     `analysis_mode: local_fallback_jieba_textrank` 與 `needs_reanalysis: true`，
     之後可以篩選這些筆數，等 API 恢復後用 Gemini 重新分析。
4. **文字/圖片優先權判斷**：文案文字達到 `TEXT_LENGTH_SKIP_IMAGES_THRESHOLD`（預設 800 字）
   視為已足夠，直接跳過圖片只用文字分析；文字偏少時才讀取圖片一併分析。
5. **輸出到 MongoDB（新增/insert 模式）**：以「專案路徑」當唯一 key，
   已經寫入過的專案不會重複分析，程式中斷後重新執行會自動跳過已完成的專案。
6. **CSV 備份**：每分析完一筆就即時 append 一行到 CSV，作為 MongoDB 之外的備援。
7. **斷點續跑 / 錯誤記錄**：無論是預期內的錯誤還是完全未預期的例外，
   都會把「目前處理到哪個專案」寫進 `checkpoint.json` 與 `error_log.txt`，
   下次重新執行時可以直接接續，不用從頭跑。
8. **修正 Colab 常見的 localhost timeout 問題**：自動清除殘留的 proxy 環境變數
   （`HTTP_PROXY` / `HTTPS_PROXY` 等），避免 Gemini API 請求被誤導到 localhost 假代理而逾時。

> ⚠️ 使用前請先在 Colab 左側 🔑 Secrets 設定好：
> `GEMINI_API_KEY_1`、`GEMINI_API_KEY_2`、`GEMINI_API_KEY_3`、`MONGODB_URI`
> 並把下面 `BASE_DIR` 換成你實際的 Google Drive 路徑。


In [ ]:
# ========================================
# Step 0：安裝套件
# ========================================
!pip install -q google-generativeai pymongo pillow jieba


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 20.4 MB/s eta 0:00:00


In [ ]:
# ========================================
# Step 1：掛載 Drive + 基本路徑設定
# ========================================
from google.colab import drive
drive.mount('/content/drive')

import os

# 🔧 修正常見的 Colab timeout 元兇：Colab 有時會殘留指向 localhost 的
# HTTP_PROXY / HTTPS_PROXY 環境變數（給 Drive、BigQuery 等內部連線用），
# 若 google-generativeai 的底層 requests/httplib 也吃到這些變數，
# 就會把打 Gemini API 的請求也導去 localhost 的假代理，進而 read timeout。
# （對應到你剛才看到的錯誤：HTTPConnectionPool(host='localhost', port=xxxxx): Read timed out）
for _proxy_var in ('HTTP_PROXY', 'HTTPS_PROXY', 'http_proxy', 'https_proxy', 'ALL_PROXY', 'all_proxy'):
    if _proxy_var in os.environ:
        print(f"⚠️ 偵測到殘留的 proxy 環境變數 {_proxy_var}={os.environ[_proxy_var]}，"
              f"可能導致連線被導到 localhost 而 timeout，已移除。")
        os.environ.pop(_proxy_var, None)

# 👉 請改成你實際的「ZecZec Group Data」路徑（掛載後的路徑）
BASE_DIR = '/content/drive/MyDrive/ZecZec_Group_Data'

# 輸出備份用的檔案（建議也放在 Drive，這樣 Colab 斷線也不會遺失）
CSV_BACKUP_PATH   = '/content/drive/MyDrive/ZecZec_keyword_results_backup.csv'
CHECKPOINT_PATH   = '/content/drive/MyDrive/ZecZec_checkpoint.json'
ERROR_LOG_PATH    = '/content/drive/MyDrive/ZecZec_error_log.txt'

# 每個專案最多送幾張圖片給 Gemini（避免圖片太多爆 token / 太慢）
MAX_IMAGES_PER_PROJECT = 8
# 圖片送出前先縮圖到多大（長邊像素），避免佔用太多 token / payload 太大導致 timeout
IMAGE_MAX_SIDE = 768
# 圖片壓縮成 JPEG 的品質（1-95），數字越小檔案越小、傳輸越不容易 timeout
IMAGE_JPEG_QUALITY = 80

# 🔋 額度保護開關：
# True（預設）→ 正常打 Gemini，配合下面的斷路器保護額度
# False → 完全不打 Gemini，全部用本地 Jieba + TextRank 處理（零 API 消耗）。
#         當你確定額度已經用完、想先把剩下的專案骨架資料跑完時，把這裡改成 False 即可。
#         這些資料會被標記 needs_reanalysis=True，之後額度恢復可以用 Step 10 的補跑功能重新分析。
USE_GEMINI_LAYER = True
# 🔌 斷路器冷卻秒數：一旦偵測到「所有 Key」都因限速失敗，就封鎖 Gemini 呼叫這麼久，
#    這段時間內所有專案都直接走本地備援（零 API 呼叫），避免對已經耗盡的額度繼續重試、白白浪費，
#    甚至可能被 Google 判定為濫用而停權。冷卻時間到了才會自動再嘗試一次 Gemini。
CIRCUIT_BREAKER_COOLDOWN_SECONDS = 1800  # 30 分鐘
# 呼叫 Gemini 的逾時秒數（圖片多的話建議拉長一點）
REQUEST_TIMEOUT_SECONDS = 60
# 文字/圖片優先權判斷：cleaned 文字達到這個字數以上，就視為「文案已經很充足」，
# 直接跳過圖片、只用文字分析（省 payload、降低 timeout 機率）。
# 反之文字偏少（例如圖多字少的專案），才會讀取圖片一併分析。
TEXT_LENGTH_SKIP_IMAGES_THRESHOLD = 1400

print("BASE_DIR 是否存在：", os.path.exists(BASE_DIR))


Mounted at /content/drive
BASE_DIR 是否存在： True


In [ ]:
# ========================================
# Step 2：設定 3 組 Gemini API Key 輪替（改用原始 REST API 直接呼叫，
#         不透過 google-generativeai SDK，避開它在 Colab 裡的內部本地轉發機制
#         ——這正是你一直卡在 localhost timeout 的元兇，是 SDK 的已知 bug，不是你的 Key 有問題）
# ========================================
import requests
from google.colab import userdata
import time

API_KEY_NAMES = ['GEMINI_API_KEY', 'GEMINI_API_KEY_2', 'GEMINI_API_KEY_3']

API_KEYS = []
for name in API_KEY_NAMES:
    try:
        key = userdata.get(name)
        if key:
            API_KEYS.append(key)
            print(f"  ✅ {name} 讀取成功（長度 {len(key)}）")
        else:
            print(f"  ⚠️ {name} 讀取到空值")
    except Exception as e:
        # 不再靜默吞掉錯誤：Colab Secrets 常見問題是「授權彈窗還沒點同意」，
        # 印出真實錯誤原因才看得出來是哪把 Key、什麼原因失敗
        print(f"  ❌ {name} 讀取失敗，真實錯誤原因：{repr(e)}")

if not API_KEYS:
    raise RuntimeError("❌ 找不到任何 Gemini API Key，請先在左側 🔑 Secrets 設定 GEMINI_API_KEY_1 / 2 / 3")

if len(API_KEYS) < len(API_KEY_NAMES):
    print(f"\n⚠️ 只成功載入 {len(API_KEYS)}/{len(API_KEY_NAMES)} 把 Key。"
          f"如果剛剛有 Key 授權彈窗跳出來才點同意，請重新執行這個 cell 一次再繼續。")

print(f"\n✅ 已載入 {len(API_KEYS)} 組 Gemini API Key，將輪流使用")

_current_key_idx = 0
MODEL_NAME = 'gemini-3.5-flash-lite'  # 沿用原本使用的模型
GEMINI_REST_ENDPOINT = "https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent"

def current_api_key():
    return API_KEYS[_current_key_idx]

def rotate_key(reason=""):
    """切換到下一把 Key（輪替使用）"""
    global _current_key_idx
    _current_key_idx = (_current_key_idx + 1) % len(API_KEYS)
    print(f"      [切換] 🔄 換用第 {_current_key_idx + 1} 組 API Key（原因：{reason}）")

# 先用第一把 Key 做一次最小請求，確認 REST 連線正常（直接連 Google，不透過 SDK 的本地轉發）
try:

    _test_resp = requests.post(
        GEMINI_REST_ENDPOINT.format(model=MODEL_NAME),
        params={"key": current_api_key()},
        json={
            "contents": [{"parts": [{"text": "回覆「ok」"}]}],
            "safetySettings": custom_safety_settings
           },
        timeout=20
    )
    # 準備你要放寬的安全審查設定（全部設為 BLOCK_NONE，也就是不封鎖）


    if _test_resp.status_code == 200:
        print("✅ REST API 連線測試成功！（已繞過 SDK 在 Colab 的本地轉發機制）")
    else:
        print(f"⚠️ REST API 測試回應狀態碼 {_test_resp.status_code}：{_test_resp.text[:200]}")
except Exception as e:
    print(f"⚠️ REST API 連線測試發生例外：{e}")


  ✅ GEMINI_API_KEY 讀取成功（長度 39）
  ✅ GEMINI_API_KEY_2 讀取成功（長度 53）
  ✅ GEMINI_API_KEY_3 讀取成功（長度 53）

✅ 已載入 3 組 Gemini API Key，將輪流使用
✅ REST API 連線測試成功！（已繞過 SDK 在 Colab 的本地轉發機制）


In [ ]:
# ========================================
# Step 3：MongoDB 連線設定（新增 / insert 模式）
# ========================================
from pymongo import MongoClient, errors as mongo_errors

try:
    MONGODB_URI = userdata.get('MONGODB_URI')
except Exception:
    MONGODB_URI = None

if not MONGODB_URI:
    # 沒設定 Secrets 的話，也可以直接把連線字串貼在這裡（僅本機測試用，不建議上傳到公開notebook）
    MONGODB_URI = "mongodb+srv://<user>:<password>@<cluster>.mongodb.net/?retryWrites=true&w=majority"

mongo_client = MongoClient(MONGODB_URI)
db = mongo_client['zeczec_db']
collection = db['keyword_analysis']

# 用 project_id（資料夾相對路徑）當唯一索引，重複寫入會直接被擋掉，達成「新增模式」
collection.create_index('project_id', unique=True)

# 啟動時先把「已經成功分析過」的 project_id 全部撈出來，之後掃描資料夾時可以快速跳過
done_project_ids = set(collection.distinct('project_id'))
print(f"✅ MongoDB 連線成功，目前已有 {len(done_project_ids)} 筆專案分析結果")


✅ MongoDB 連線成功，目前已有 1805 筆專案分析結果


In [ ]:
# ========================================
# Step 4：OCR 清洗 + Prompt（保留原本內容，僅新增「圖片參考」說明）
# ========================================
import re

def clean_ocr_noise(text):
    """清洗 OCR 亂碼"""
    text = re.sub(r'[^\u4e00-\u9fff\w\s，。！？、：；「」【】（）\-]', ' ', text)
    text = re.sub(r'(?<!\w)[a-zA-Z]{1,2}(?!\w)', ' ', text)  # 移除孤立英文字母
    text = re.sub(r'(.{2,4})\1{2,}', r'\1', text)             # 移除重複詞
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def build_prompt(cleaned_text, has_images):
    """組合 Prompt，文字部分完全沿用原本 Notebook 的版本"""

    image_note = ""
    if has_images:
        image_note = (
            "\n- 本次額外提供了此募資專案的產品圖片，請將圖片中呈現的「產品外觀、包裝、"
            "色彩、材質、使用情境」也一併納入判斷，讓 style、core_features、"
            "trend_keywords 與 Long-tail_Keywords 更貼近圖片中實際呈現的樣貌，"
            "而不要只依賴文字內容。\n"
        )

    prompt = f"""你是一個專業的群眾募資分析師。
以下是一個募資專案的內文（可能含有OCR識別錯誤）{"，並附上該專案的產品圖片" if has_images else ""}。

請注意：
- 忽略退款條款、物流出貨、客服聯絡、法律聲明等行政與免責文字。
- 專注於「這個產品的核心功能、應用場景、設計風格、獨特賣點 (USP) 或背景故事」。
- trend_keywords 必須是「台灣消費者在網購或尋找此類產品時，真正會輸入 Google 的大方向通用關鍵字」（例如：恐怖遊戲、獨立遊戲、解謎桌遊）。請絕對避免將專案特有的劇情、專有名詞或冷門詞彙（如：東歐、精神實驗）當作搜尋關鍵字，以免搜尋量為零。
- 請嚴格確保所有輸出的內容與關鍵字，都必須是『台灣在地化用語』與『繁體中文』，絕對不可出現簡體字。
- 請提供 5 個『長尾關鍵字（Long-tail Keywords）』。不要只輸出『寵物玩具』這種大範圍詞彙，請務必結合產品的『特殊材質（如貓草）』、『功能（如嗅聞、放電）』或『解決的痛點（如焦慮、無聊）』來生成，以符合實際消費者的搜尋習慣。{image_note}

請完全「根據下方的專案內文」進行客觀判斷，絕不可自行發明或套用無關的風格。只輸出以下JSON格式，不要輸出其他任何說明文字：

{{
  "product_type": "<填入精準的產品類型，例如判斷是桌遊、家電或背包>",
  "core_features": ["<填入實際提及的核心特色1>", "<填入實際提及的核心特色2>", "<填入實際提及的核心特色3>"],
  "style": "<萃取內文提及的視覺設計或整體風格，如無明確提及請填『無』>",
  "target_audience": "<填入最精準的目標客群描述>",
  "trend_keywords": ["<通用關鍵字1>", "<通用關鍵字2>", "<通用關鍵字3>"],
  "Long-tail_Keywords": ["<長尾關鍵字1>", "<長尾關鍵字2>", "<長尾關鍵字3>", "<長尾關鍵字4>", "<長尾關鍵字5>"]
}}

專案內文：
{cleaned_text}"""

    return prompt

print('✅ Prompt 與清洗函式定義完成（內容與原本一致，僅新增圖片說明段落）')


✅ Prompt 與清洗函式定義完成（內容與原本一致，僅新增圖片說明段落）


In [ ]:
# ========================================
# Step 5：讀取專案圖片（縮圖 + 壓縮成 JPEG bytes，減少 payload 大小避免 timeout）
# ========================================
from PIL import Image
import io

IMAGE_EXTS = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')

def load_project_images(image_dirs, max_images=MAX_IMAGES_PER_PROJECT,
                         max_side=IMAGE_MAX_SIDE, jpeg_quality=IMAGE_JPEG_QUALITY):
    """
    從一個或多個圖片資料夾中讀取圖片，縮圖後壓縮成 JPEG bytes 回傳。
    直接丟 PIL Image 物件給 SDK 常常會用未壓縮/PNG 格式傳送，payload 很大、
    在 Colab 的網路環境下很容易 timeout，所以這裡手動轉成小一點的 JPEG bytes。
    回傳格式：[{"mime_type": "image/jpeg", "data": <bytes>}, ...]
    """
    images = []
    for image_dir in image_dirs:
        if not os.path.isdir(image_dir):
            continue
        for fname in sorted(os.listdir(image_dir)):
            if len(images) >= max_images:
                return images
            if not fname.lower().endswith(IMAGE_EXTS):
                continue
            fpath = os.path.join(image_dir, fname)
            try:
                img = Image.open(fpath).convert('RGB')
                img.thumbnail((max_side, max_side))
                buf = io.BytesIO()
                img.save(buf, format='JPEG', quality=jpeg_quality, optimize=True)
                images.append({"mime_type": "image/jpeg", "data": buf.getvalue()})
            except Exception as e:
                print(f"      ⚠️ 無法讀取圖片 {fpath}：{e}")
    return images

print('✅ 圖片讀取函式定義完成（已改為壓縮 JPEG bytes，降低 timeout 機率）')


✅ 圖片讀取函式定義完成（已改為壓縮 JPEG bytes，降低 timeout 機率）


In [ ]:
# ========================================
# Step 6：第二層防禦 — 本地端傳統 NLP 備援（Jieba + TextRank）
# ========================================
import jieba
import jieba.analyse

# 保留較能代表產品特徵的詞性：名詞、動名詞、動詞、專有名詞、地名、英文
_FALLBACK_ALLOW_POS = ('n', 'vn', 'v', 'nz', 'ns', 'eng')

def local_fallback_analysis(cleaned_text, topK=15):
    """
    第二層防禦（On-premise Classical NLP）：
    當第一層（Gemini + 多 Key 輪替）完全打不通時，改由本地端運行
    Jieba 斷詞 + TextRank 權重演算法，在完全離線的狀態下，
    動態分析募資內文之高頻核心名詞，抓取最具代表性的特徵詞彙作為關鍵字，
    確保關鍵字產出不中斷、核心指標仍可持續計算。
    """
    print("      ⚠️ AI 連線異常，目前啟用本地精簡分析模式，語意深度評估可能受限。")

    try:
        keywords = jieba.analyse.textrank(
            cleaned_text, topK=topK, withWeight=False, allowPOS=_FALLBACK_ALLOW_POS
        )
    except Exception as e:
        keywords = []
        print(f"      ⚠️ 本地 NLP 分析也發生例外：{e}")

    if not keywords:
        keywords = ["無法擷取關鍵字"]

    trend_keywords = keywords[:3]
    long_tail = keywords[3:8] if len(keywords) > 3 else keywords[:5]

    return {
        "product_type": "（本地模式無法判斷，待 API 恢復後建議重新分析）",
        "core_features": keywords[:3],
        "style": "（本地模式無法判斷）",
        "target_audience": "（本地模式無法判斷）",
        "trend_keywords": trend_keywords,
        "Long-tail_Keywords": long_tail,
        "analysis_mode": "local_fallback_jieba_textrank",
        "needs_reanalysis": True,  # 之後可用這個欄位篩出「等 API 恢復後要重新用 Gemini 分析」的專案
    }

print("✅ 第二層防禦（本地 Jieba + TextRank 備援）定義完成")


/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
/usr/local/lib/python3.12/dist-packages/jieba/posseg/__init__.py:16: SyntaxWarning: invalid escape sequence '\.'
  re_skip_detail = re.compile("([\.0-9]+|[a-zA-Z0-9]+)")
/usr/local/lib/python3.12/dist-packages/jieba/posseg/__init__.py:17: SyntaxWarning: invalid escape sequence '\.'
  re_han_internal = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._]+)")
/usr/local/lib/python3.12/dist-packages/jieba/posseg/__init__.py:18: SyntaxWarning: invalid escape sequence '\s'
  re_skip_internal = re.compil

✅ 第二層防禦（本地 Jieba + TextRank 備援）定義完成


In [ ]:
# ========================================
# Step 7：第一層防禦 — 多金鑰輪替（Key Failover）+ 額度斷路器（Circuit Breaker），
#         全部打不通時自動交給第二層（本地 NLP 備援）
# ========================================
import json, time

def _is_timeout_error(error_msg):
    e = error_msg.lower()
    return ("timed out" in e or "timeout" in e or "connection aborted" in e
            or "connection reset" in e or "deadline exceeded" in e)


def _is_quota_error(error_msg):
    e = error_msg.lower()
    return "429" in error_msg or "quota" in e or "resource_exhausted" in e


def _is_blocked_content_error(error_msg):
    """
    偵測 Gemini 安全過濾機制擋下的內容（例如 PROHIBITED_CONTENT）。
    這種錯誤跟額度/連線完全無關，是『這段文字本身』被判定違反安全政策，
    不管換哪把 Key、重試幾次都一定會是同樣結果，所以偵測到就直接放棄重試，
    避免浪費 API 呼叫次數。
    """
    e = error_msg.lower()
    return "prohibited_content" in e or "block_reason" in e or "blocked prompt" in e


# 第一層：Key 切換後只短暫等待這麼久（秒）就重試，達成「近乎無縫」切換
KEY_FAILOVER_WAIT_SECONDS = 1.5
# 圖片最多降級幾輪（每輪張數砍半：例如 8 -> 4 -> 2 -> 1 -> 0），
# 每一輪都會把所有 Key 各嘗試一次（只有在還沒觸發斷路器時才會跑到這裡）
MAX_IMAGE_DEGRADE_ROUNDS = 3

# 斷路器狀態：在這個時間點之前，完全不呼叫 Gemini，直接用本地備援
_quota_circuit_open_until = 0.0


import base64

def _call_gemini_once(cleaned_text, images):
    """
    用『目前這把 Key』呼叫一次 Gemini，回傳 (成功結果 or None, 錯誤訊息 or None)。
    直接用 requests 打 REST API，不透過 google-generativeai SDK
    （SDK 在 Colab 裡會自動改走內部的本地轉發機制，那個機制不穩定，是造成
    localhost timeout 的元兇；改用原始 REST 呼叫可以完全繞開這個問題）。
    """
    prompt = build_prompt(cleaned_text, has_images=len(images) > 0)

    parts = [{"text": prompt}]
    for img in images:
        parts.append({
            "inline_data": {
                "mime_type": img["mime_type"],
                "data": base64.b64encode(img["data"]).decode("utf-8")
            }
        })

    payload = {
        "contents": [{"parts": parts}],
        "generationConfig": {
            "responseMimeType": "application/json",
            "temperature": 0.1,

        }
    }

    try:
        resp = requests.post(
            GEMINI_REST_ENDPOINT.format(model=MODEL_NAME),
            params={"key": current_api_key()},
            json=payload,
            timeout=REQUEST_TIMEOUT_SECONDS
        )
    except requests.exceptions.RequestException as e:
        # 網路層級的例外：timeout、連線中斷等，訊息格式跟以前 SDK 丟出來的很類似，
        # 沿用同一套 _is_timeout_error() 判斷邏輯即可辨識
        return None, str(e)

    if resp.status_code != 200:
        # 429 限速就會落在這裡，錯誤訊息保留狀態碼方便 _is_quota_error() 判斷
        return None, f"{resp.status_code} {resp.text[:500]}"

    try:
        data = resp.json()
    except Exception as e:
        return None, f"回應不是合法 JSON：{e} / raw={resp.text[:300]}"

    candidates = data.get('candidates')
    if not candidates:
        feedback = data.get('promptFeedback', {})
        block_reason = feedback.get('blockReason', 'UNKNOWN')
        return None, f"Invalid operation: blocked prompt, block_reason: {block_reason} PROHIBITED_CONTENT"

    try:
        text = candidates[0]['content']['parts'][0]['text']
        return json.loads(text), None
    except (KeyError, IndexError, json.JSONDecodeError) as e:
        return None, f"回應內容解析失敗：{e} / raw={str(data)[:300]}"


def _try_all_keys_once(cleaned, images):
    """
    把所有 Key 各嘗試一次（除非中途遇到內容安全封鎖，那種情況會立刻停止，不再浪費呼叫）。
    回傳 (成功結果 or None, 這一輪是否『全部』都是因為限速而失敗, 是否被安全機制封鎖內容)
    """

    # 準備好免死金牌（安全設定），先放旁邊備用
    custom_safety_settings = [
        {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
        {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
        {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
        {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"}
    ]
    all_failed_were_quota = True
    for key_idx in range(len(API_KEYS)):
        print(f"      [除錯] 📡 發送請求中...（第 {key_idx + 1}/{len(API_KEYS)} 把 Key，圖片 {len(images)} 張）")
        result, error_msg = _call_gemini_once(cleaned, images)

        if error_msg is None:
            print("      [除錯] 📥 成功收到回傳！（第一層防禦成功）")
            result['analysis_mode'] = 'gemini'
            return result, False, False

        if _is_blocked_content_error(error_msg):
            print("      [除錯] 🚫 內容被 Gemini 安全機制封鎖（PROHIBITED_CONTENT），"
                  "與額度/連線無關，換 Key 或重試也不會成功，直接跳過不再重試")
            return None, False, True

        if _is_quota_error(error_msg):
            rotate_key(reason="429 限速")
            time.sleep(KEY_FAILOVER_WAIT_SECONDS)  # 快速切換，不長時間 backoff
        elif _is_timeout_error(error_msg):
            all_failed_were_quota = False
            rotate_key(reason="連線逾時/中斷")
            time.sleep(KEY_FAILOVER_WAIT_SECONDS)
        else:
            all_failed_were_quota = False
            print(f"      [除錯] ❌ 發生未預期錯誤：{error_msg}")
            time.sleep(2)

    return None, all_failed_were_quota, False


def extract_keywords_multimodal(raw_text, images):
    """
    第一層防禦（Failover Key Pool + 額度斷路器）：
      - 若 USE_GEMINI_LAYER = False，完全不打 API，直接用本地備援（零額度消耗）。
      - 若目前正處於斷路器冷卻期間（代表前面已經偵測到額度用盡），也直接用本地備援，
        不再浪費任何一次 API 呼叫，避免對已耗盡的額度繼續重試（這樣做除了浪費，
        還可能被 Google 判定為濫用而讓 Key 被停權）。
      - 否則對每一把 API Key 各嘗試一次；429 限速或連線逾時就在
        KEY_FAILOVER_WAIT_SECONDS 秒內切換下一把繼續。若這一輪『所有 Key』都因限速失敗，
        代表額度多半是共用、真的滿了：**立刻開啟斷路器**，往後 CIRCUIT_BREAKER_COOLDOWN_SECONDS
        秒內的所有專案都直接走本地備援，不會再逐筆重新嘗試、逐筆燒額度。
        （若是連線逾時而非限速，才會走原本的圖片降級重試邏輯，因為那通常跟額度無關。）

    第二層防禦（On-premise Classical NLP）：
      第一層被跳過或完全失敗時，觸發 local_fallback_analysis()，
      改用本地 Jieba + TextRank 在離線狀態下抓取關鍵字，維持這筆資料的核心指標仍能產出，
      並標記 needs_reanalysis=True，供額度恢復後用 Step 10 的補跑功能重新分析。
    """
    global _quota_circuit_open_until
    cleaned = clean_ocr_noise(raw_text)

    if not USE_GEMINI_LAYER:
        print("      [除錯] 🔋 USE_GEMINI_LAYER=False，直接使用本地備援（不消耗任何額度）")
        return local_fallback_analysis(cleaned)

    now = time.time()
    if now < _quota_circuit_open_until:
        remaining = int(_quota_circuit_open_until - now)
        print(f"      [除錯] ⏭️ 斷路器冷卻中（還剩約 {remaining} 秒），直接使用本地備援，不呼叫 API")
        return local_fallback_analysis(cleaned)

    current_images = list(images)

    for round_idx in range(MAX_IMAGE_DEGRADE_ROUNDS + 1):
        result, round_all_quota, blocked = _try_all_keys_once(cleaned, current_images)
        if result is not None:
            return result

        if blocked:
            # 內容被安全機制封鎖：與額度無關，直接進本地備援，不開斷路器、不再重試
            fallback = local_fallback_analysis(cleaned)
            fallback['blocked_by_safety_filter'] = True
            return fallback

        if round_all_quota:
            # 所有 Key 都因限速失敗 -> 判定額度已耗盡，立刻開啟斷路器，不再逐筆重試
            _quota_circuit_open_until = time.time() + CIRCUIT_BREAKER_COOLDOWN_SECONDS
            print(f"      [除錯] 🚨 所有 Key 皆遭限速，判定額度已耗盡，"
                  f"開啟斷路器 {CIRCUIT_BREAKER_COOLDOWN_SECONDS} 秒（期間內都直接用本地備援）")
            break

        # 走到這裡代表是「逾時/連線中斷」而非限速，跟圖片 payload 較有關，
        # 才進行圖片降級後重試
        if len(current_images) == 0:
            break
        current_images = current_images[: max(0, len(current_images) // 2)]
        print(f"      [除錯] ⏱️ 這一輪所有 Key 都逾時失敗，下一輪改用 {len(current_images)} 張圖片")

    # ---------- 第一層失敗（或斷路器已開啟）-> 觸發第二層：本地 NLP 備援 ----------
    return local_fallback_analysis(cleaned)

print("✅ 第一層防禦（多金鑰輪替 + 額度斷路器 + 圖片降級）定義完成，並已串接第二層本地 NLP 備援")


✅ 第一層防禦（多金鑰輪替 + 額度斷路器 + 圖片降級）定義完成，並已串接第二層本地 NLP 備援


In [ ]:
# ========================================
# Step 7：遞迴掃描 ZecZec Group Data，找出所有專案
# ========================================
def find_projects(base_dir):
    """
    不假設固定層數的資料夾結構，只要某資料夾內含有 *content.txt，
    就視為一個專案（例如 TF1_peakstw 內有 TF1_content.txt + TF1_images/）。
    """
    projects = []
    for root, dirs, files in os.walk(base_dir):
        content_files = [f for f in files if f.endswith('content.txt')]
        if not content_files:
            continue

        image_dirs = [
            os.path.join(root, d) for d in dirs
            if d.lower().endswith('images')
        ]

        project_id = os.path.relpath(root, base_dir)  # 當作唯一識別碼，同時保留完整層級資訊
        projects.append({
            'project_id': project_id,
            'path': root,
            'content_files': [os.path.join(root, f) for f in content_files],
            'image_dirs': image_dirs,
        })

    return projects

all_projects = find_projects(BASE_DIR)
print(f"📂 掃描完成，共找到 {len(all_projects)} 個專案")


📂 掃描完成，共找到 2104 個專案


In [ ]:
# ========================================
# Step 8：checkpoint / CSV / 錯誤紀錄 輔助函式
# ========================================
import csv
import traceback
from datetime import datetime

def save_checkpoint(project_id, status, extra=None):
    """把目前處理到哪個專案寫進 checkpoint.json，方便斷線後知道進度"""
    data = {
        'last_project_id': project_id,
        'status': status,          # 'success' / 'error' / 'skipped'
        'timestamp': datetime.now().isoformat(),
        'total_done_in_mongo': len(done_project_ids),
    }
    if extra:
        data['extra'] = extra
    with open(CHECKPOINT_PATH, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def log_error(project_id, error_text):
    with open(ERROR_LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(f"[{datetime.now().isoformat()}] 專案：{project_id}\n{error_text}\n{'-'*60}\n")


def append_to_csv(result_dict):
    """即時 append 一行到 CSV 備份檔（第一次寫入時自動建立表頭）"""
    file_exists = os.path.exists(CSV_BACKUP_PATH)
    # 把 list 欄位轉成字串，方便存 CSV
    row = {}
    for k, v in result_dict.items():
        row[k] = "、".join(v) if isinstance(v, list) else v

    with open(CSV_BACKUP_PATH, 'a', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


def save_to_mongo(result_dict):
    """新增模式：project_id 已存在就會丟 DuplicateKeyError，直接視為已完成、略過"""
    try:
        collection.insert_one(result_dict)
        return True
    except mongo_errors.DuplicateKeyError:
        return False

print("✅ checkpoint / CSV / 錯誤紀錄函式定義完成")


✅ checkpoint / CSV / 錯誤紀錄函式定義完成


In [ ]:
# ========================================
# Step 9：主流程（可斷點續跑、跑完全部 ZecZec Group Data）
# ========================================
print(f"🚀 開始處理，共 {len(all_projects)} 個專案，已完成 {len(done_project_ids)} 個（將自動跳過）\n")

processed_this_run = 0
skipped_this_run = 0
failed_this_run = 0

for project in all_projects:
    project_id = project['project_id']

    # 1) 已經在 MongoDB 有紀錄 -> 直接跳過（達成「斷掉不用重跑」）
    if project_id in done_project_ids:
        skipped_this_run += 1
        continue

    print(f"\n🔄 處理中：{project_id}")

    try:
        # 2) 讀取所有 *content.txt（含 ocr_content.txt）
        combined_text = ""
        for fpath in project['content_files']:
            with open(fpath, 'r', encoding='utf-8') as f:
                combined_text += f.read() + "\n"

        text_length = len(combined_text.strip())
        if text_length == 0:
            print("  ⚠️ 找不到有效文字內容，略過此專案")
            save_checkpoint(project_id, 'skipped', extra='無文字內容')
            skipped_this_run += 1
            continue

        # 3) 讀取圖片（依「文字是否充足」決定要不要分析圖片）
        #    - 文字夠多 -> 文案本身資訊已足夠，跳過圖片，省 payload、降低 timeout 機率
        #    - 文字偏少 -> 就算圖片很多張，也要讀取圖片一併分析，補足文字資訊不足的部分
        if text_length >= TEXT_LENGTH_SKIP_IMAGES_THRESHOLD:
            images = []
            print(f"  📄 文字 {text_length} 字（已達 {TEXT_LENGTH_SKIP_IMAGES_THRESHOLD} 字門檻，視為充足）"
                  f"，⏭️ 跳過圖片，只用文字分析")
        else:
            images = load_project_images(project['image_dirs'])
            print(f"  📄 文字僅 {text_length} 字（未達 {TEXT_LENGTH_SKIP_IMAGES_THRESHOLD} 字門檻）"
                  f"，🖼️ 讀取 {len(images)} 張圖片一併分析，⏳ Gemini 分析中...")

        # 4) 呼叫 Gemini（文字+圖片）
        result = extract_keywords_multimodal(combined_text, images)

        if 'error' in result:
            failed_this_run += 1
            log_error(project_id, result.get('error', '未知錯誤'))
            save_checkpoint(project_id, 'error', extra=result.get('error'))
            print(f"  ❌ 分析失敗：{result.get('error')}")
            continue

        # 5) 補上層級資訊（用資料夾路徑自動拆解，取代原本寫死 4 層的作法）
        parts = project_id.split(os.sep)
        result['project_id'] = project_id
        result['project_path_parts'] = parts
        result['Project_Name'] = parts[-1]
        result['image_count'] = len(images)
        result['analyzed_at'] = datetime.now().isoformat()

        # 6) 寫入 MongoDB（新增模式）+ CSV 備份
        inserted = save_to_mongo(result)
        if inserted:
            done_project_ids.add(project_id)
            append_to_csv(result)
            processed_this_run += 1
            print(f"  ✅ 成功！trend_keywords：{result.get('trend_keywords')}")
        else:
            skipped_this_run += 1
            print("  ⚠️ MongoDB 中已存在，視為完成，略過")

        save_checkpoint(project_id, 'success')

        time.sleep(4)  # 稍作間隔，降低被限速機率

    except Exception as e:
        # 🔥 完全未預期的例外（例如圖片壞檔、網路斷線、Drive 掛載掉了等）
        # 記錄目前進度後繼續處理下一個專案，不讓整個流程當機
        failed_this_run += 1
        err_text = traceback.format_exc()
        log_error(project_id, err_text)
        save_checkpoint(project_id, 'error', extra=str(e))
        print(f"  ❌ 未預期錯誤，已記錄進度並跳過此專案：{e}")
        continue

print("\n========== 本次執行結束 ==========")
print(f"新完成：{processed_this_run}　略過（已完成）：{skipped_this_run}　失敗：{failed_this_run}")
print(f"MongoDB 目前累計筆數：{len(done_project_ids)}")
print(f"CSV 備份位置：{CSV_BACKUP_PATH}")
print(f"Checkpoint 位置：{CHECKPOINT_PATH}")
print(f"錯誤紀錄位置：{ERROR_LOG_PATH}")


🚀 開始處理，共 2104 個專案，已完成 2104 個（將自動跳過）


========== 本次執行結束 ==========
新完成：0　略過（已完成）：2104　失敗：0
MongoDB 目前累計筆數：2104
CSV 備份位置：/content/drive/MyDrive/ZecZec_keyword_results_backup.csv
Checkpoint 位置：/content/drive/MyDrive/ZecZec_checkpoint.json
錯誤紀錄位置：/content/drive/MyDrive/ZecZec_error_log.txt


---
## Step 10（維護用，額度恢復後再執行）：補跑本地備援的資料

上面主流程如果因為額度耗盡（斷路器開啟）或 `USE_GEMINI_LAYER=False`，
有些專案會是用**本地 Jieba + TextRank** 產出的，這些資料在 MongoDB 裡會被標記
`needs_reanalysis: true`，品質比 Gemini 差一些。

**等你確認額度已經恢復（例如隔天、或檢查 AI Studio 額度頁面），
再執行下面這個 cell**，它只會挑出這些被標記的專案，重新用 Gemini 分析一次
（並且是用 `update`覆蓋，不是新增，也不會動到已經是 `analysis_mode: gemini` 的正常資料）。

執行前記得先把上面 Step 1 的 `USE_GEMINI_LAYER` 改回 `True`。


In [ ]:
# ========================================
# Step 10：補跑 — 只重新分析被標記 needs_reanalysis=True 的專案（維護用，額度恢復後才執行）
# ========================================
def update_in_mongo(project_id, result_dict):
    """覆蓋更新既有文件（用在補跑本地備援資料時），而不是新增"""
    collection.update_one({'project_id': project_id}, {'$set': result_dict})


def reanalyze_needs_reanalysis(limit=None):
    to_fix = list(collection.find({'needs_reanalysis': True}))
    if limit:
        to_fix = to_fix[:limit]

    print(f"🔧 找到 {len(to_fix)} 筆待補跑的專案（analysis_mode=local_fallback_jieba_textrank）\n")

    fixed, still_failed = 0, 0
    for doc in to_fix:
        project_id = doc['project_id']
        project_path = os.path.join(BASE_DIR, project_id)
        content_files = [f for f in os.listdir(project_path) if f.endswith('content.txt')] \
            if os.path.isdir(project_path) else []

        if not content_files:
            print(f"  ⚠️ 找不到原始資料夾/檔案，略過：{project_id}")
            still_failed += 1
            continue

        combined_text = ""
        for fname in content_files:
            with open(os.path.join(project_path, fname), 'r', encoding='utf-8') as f:
                combined_text += f.read() + "\n"

        text_length = len(combined_text.strip())
        image_dirs = [os.path.join(project_path, d) for d in os.listdir(project_path)
                      if os.path.isdir(os.path.join(project_path, d)) and d.lower().endswith('images')]

        if text_length >= TEXT_LENGTH_SKIP_IMAGES_THRESHOLD:
            images = []
        else:
            images = load_project_images(image_dirs)

        print(f"🔄 補跑中：{project_id}（文字 {text_length} 字，圖片 {len(images)} 張）")
        result = extract_keywords_multimodal(combined_text, images)

        if result.get('analysis_mode') == 'gemini':
            result['project_id'] = project_id
            result['image_count'] = len(images)
            result['analyzed_at'] = datetime.now().isoformat()
            result['needs_reanalysis'] = False
            update_in_mongo(project_id, result)
            fixed += 1
            print(f"  ✅ 補跑成功！trend_keywords：{result.get('trend_keywords')}")
        else:
            still_failed += 1
            print("  ⚠️ 這次還是沒能用 Gemini 分析成功（可能額度還沒恢復），下次再試")

        time.sleep(4)

    print(f"\n========== 補跑結束 ==========")
    print(f"成功補跑：{fixed}　仍待下次補跑：{still_failed}")


# 👉 想跑的時候，取消下面這行的註解執行（可先用 limit=20 小量測試額度是否真的恢復了）
reanalyze_needs_reanalysis(limit=100)


🔧 找到 1 筆待補跑的專案（analysis_mode=local_fallback_jieba_textrank）

🔄 補跑中：出版_New_ZecZec_Dataset/預購式專案/出版/成功/PS18_metoo（文字 3465 字，圖片 0 張）
      [除錯] 📡 發送請求中...（第 1/3 把 Key，圖片 0 張）
      [除錯] 🚫 內容被 Gemini 安全機制封鎖（PROHIBITED_CONTENT），與額度/連線無關，換 Key 或重試也不會成功，直接跳過不再重試
      ⚠️ AI 連線異常，目前啟用本地精簡分析模式，語意深度評估可能受限。
  ⚠️ 這次還是沒能用 Gemini 分析成功（可能額度還沒恢復），下次再試

========== 補跑結束 ==========
成功補跑：0　仍待下次補跑：1
